# Pakistani-Context NER Model Fine-Tuning (Colab / Kaggle GPU)
**TechFest NER Challenge: Dory's Data Dive (Round 2)**

This notebook trains `xlm-roberta-large` (or fallback `xlm-roberta-base`) on the Pakistani-Context dataset.
- 20 exact entity classes
- Subword alignment with word_ids() (-100 masking)
- FP16 mixed precision
- Automatic checkpoint saving to Google Drive

In [ ]:
# 1. Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 2. Install pinned dependencies
!pip install -q transformers>=4.40.0 datasets>=2.19.0 accelerate>=0.30.0 seqeval>=1.2.2 pyyaml

In [ ]:
# 3. Mount Google Drive for persistent checkpoint saving
from google.colab import drive
drive.mount('/content/drive')

# Set output directory on Drive
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/ner_pk_models/best_ner_model"
!mkdir -p {DRIVE_OUTPUT_DIR}

In [ ]:
# 4. Launch Training
!python src/train.py \
    --model_name "xlm-roberta-base" \
    --train_file "data/processed/train.jsonl" \
    --dev_file "data/processed/dev.jsonl" \
    --output_dir {DRIVE_OUTPUT_DIR} \
    --batch_size 16 \
    --epochs 6 \
    --lr 2.5e-5 \
    --fp16

In [ ]:
# 5. Evaluate Baseline and Hybrid system
!python src/baseline.py
!python src/evaluate.py --split dev